# BOTOTOY 2.0 — YOLO11n training on Google Colab

**CSST 106 – Perception and Computer Vision** · Fruit Detection and Identification Using YOLO

This notebook fine-tunes **YOLO11n** (transfer learning from COCO) on our six fruit classes.
It uses the exact same code, dataset version and train/val split as the project repository.

**Before you start**
1. **Runtime → Change runtime type → T4 GPU**, then **Save**.
2. Run the cells **in order** (Shift+Enter). The first run asks for Google Drive access: checkpoints are saved
   to Drive so an interrupted session can be **resumed**.
3. Keep this browser tab open while training (about 2–3 hours on a T4). Colab may disconnect idle sessions.

**What you bring back:** `bototoy_training_results.zip` (≈ 10 MB). In the project folder on the laptop run
`python -m src.yolo.training_report path/to/bototoy_training_results.zip`.

> The **test split is not used here**. Model selection uses only the validation split; the test set is
> evaluated once, locally, at the end of Phase 4.

## 1. Check the GPU

In [ ]:
import torch, subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4 GPU, then run again."
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

## 2. Connect Google Drive (checkpoints survive disconnects)

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/BOTOTOY_2.0"
RUNS_DIR = f"{DRIVE_DIR}/runs"
os.makedirs(RUNS_DIR, exist_ok=True)
print("Runs will be saved in:", RUNS_DIR)

## 3. Get the project code and install Ultralytics

In [ ]:
REPO_URL = "https://github.com/imPickleRiick/BOTOTOY-2.0-Fruit-Detection-and-Identification-Using-YOLO.git"
REPO_DIR = "/content/bototoy"
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log -1 --format="Code version: %h %s"
# Same Ultralytics version as the laptop; Colab's pre-installed CUDA PyTorch is kept.
!pip install -q ultralytics==8.4.215
import ultralytics; print("Ultralytics", ultralytics.__version__)

## 4. Prepare the dataset (same files and split as the repository)
Downloads the pinned dataset (~500 MB), re-validates every image/label, and rebuilds the
**committed** train/val/test split. Takes ~3–4 minutes.

In [ ]:
!python -m src.data.download_dataset
!python -m src.data.dataset_validator | tail -n 12
!python -m src.data.dataset_splitter --from-manifest | tail -n 6

import json
summary = json.load(open("data/splits/split_summary.json"))
counts = {s: v["images"] for s, v in summary["splits"].items()}
print("Split:", counts, "| leakage-free:", summary["verification"]["leakage_free"])
assert counts == {"train": 7108, "val": 686, "test": 682}, "Split differs from the repository!"
assert summary["verification"]["leakage_free"]

## 5. Train YOLO11n
Settings come from `config/config.yaml` (image size 640, up to 50 epochs, batch 16, early-stopping patience 10,
SGD lr 0.01, seed 42). You can change them below. The best checkpoint (`best.pt`) is chosen on the
**validation** split.

*If you get "CUDA out of memory", set `BATCH = 8` and run this cell again.*

In [ ]:
EPOCHS, BATCH, IMGSZ = 50, 16, 640
RUN_NAME = "yolo11n_fruits"
!python -m src.yolo.train --device 0 --epochs {EPOCHS} --batch {BATCH} --imgsz {IMGSZ} --workers 2 --project "{RUNS_DIR}" --name {RUN_NAME} --no-copy

### 5b. Only if the session disconnected during training
Reconnect (Runtime → T4 GPU), run cells **1–4** again, then set `RESUME = True` and run this cell.
It continues from the last saved epoch on Google Drive.

In [ ]:
RESUME = False
import glob
if RESUME:
    candidates = sorted(glob.glob(f"{RUNS_DIR}/*/weights/last.pt"), key=os.path.getmtime)
    assert candidates, "No last.pt found on Drive - nothing to resume."
    LAST = candidates[-1]
    print("Resuming", LAST)
    !python -m src.yolo.train --resume "{LAST}" --no-copy
else:
    print("RESUME is False - skipping.")

## 6. Check the result and download it

In [ ]:
import glob, zipfile, json, os
runs = [d for d in glob.glob(f"{RUNS_DIR}/*") if os.path.isfile(f"{d}/weights/best.pt") and os.path.isfile(f"{d}/training_summary.json")]
assert runs, "No finished run found (best.pt + training_summary.json). Did training complete?"
RUN = max(runs, key=os.path.getmtime)
s = json.load(open(f"{RUN}/training_summary.json"))
print("Run:", RUN)
print("Validation (best epoch):", json.dumps(s.get("validation"), indent=1))

from IPython.display import Image, display
for plot in ["results.png", "confusion_matrix_normalized.png", "BoxPR_curve.png"]:
    if os.path.isfile(f"{RUN}/{plot}"):
        display(Image(filename=f"{RUN}/{plot}", width=900))

ZIP = f"{DRIVE_DIR}/bototoy_training_results.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(RUN):
        for f in files:
            if f == "last.pt":          # optimizer checkpoint, not needed for inference
                continue
            full = os.path.join(root, f)
            z.write(full, os.path.join(os.path.basename(RUN), os.path.relpath(full, RUN)))
print(f"Saved {ZIP} ({os.path.getsize(ZIP)/1e6:.1f} MB) - also on your Google Drive.")
from google.colab import files
files.download(ZIP)

## 7. Next steps on the laptop
1. Put `bototoy_training_results.zip` anywhere (e.g. Downloads).
2. In the project folder (virtual environment active):
   ```
   python -m src.yolo.training_report path/to/bototoy_training_results.zip
   python -m src.yolo.evaluate
   ```
   The first command verifies and installs `models/trained/bototoy_yolo11n.pt` and writes the training report;
   the second evaluates on the validation split.